# Fake-News Detection System Using Machine Learning
### Academic Capstone Project: End-to-End NLP & ML Pipeline
**Objective:** Build an end-to-end Machine Learning system that classifies news articles as Fake (0) or Real (1) using Natural Language Processing, TF-IDF vectorization, and five benchmark classification algorithms.

## 1. Import Libraries and Dependencies

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

# Add parent directory to path to load modular src components
sys.path.append("..")
from src.data_preprocessing import load_dataset, preprocess_dataframe, clean_text
from src.feature_engineering import build_tfidf_vectorizer, fit_and_transform_tfidf
from src.evaluate_models import evaluate_single_model, generate_comparison_report, select_best_model

sns.set_theme(style="whitegrid")
print("Environment and libraries successfully loaded!")

## 2. Exploratory Data Analysis (EDA) on WELFake Dataset

In [ ]:
data_path = "../data/WELFake_Dataset.csv"
df_raw = load_dataset(data_path)

print("Dataset Shape:", df_raw.shape)
print("\nColumns:", list(df_raw.columns))
print("\nMissing Values Summary:")
print(df_raw.isnull().sum())
print("\nClass Distribution (Raw):")
print(df_raw["label"].value_counts())

## 3. Data Cleaning and Text Preprocessing Pipeline
- Handles missing values in Title and Text
- Removes duplicate articles
- Combines Title + Text into unified input
- Strips HTML, URLs, punctuation, and non-alphabetic symbols
- Performs lowercasing, stop-word removal, and Porter Stemming

In [ ]:
df_clean = preprocess_dataframe(df_raw, apply_stemming=True)
print("Cleaned Data Dimensions:", df_clean.shape)
print("\nSample Preprocessed Data:")
df_clean[["title", "clean_text", "label"]].head(5)

## 4. Train / Test Split (Stratified 80/20)

In [ ]:
X = df_clean["clean_text"].values
y = df_clean["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set samples : {len(X_train)}")
print(f"Testing set samples  : {len(X_test)}")

## 5. TF-IDF Feature Engineering
We extract n-grams (1, 2) with sublinear term frequency scaling, fit strictly on the training set to prevent data leakage.

In [ ]:
vectorizer = build_tfidf_vectorizer(max_features=40000, ngram_range=(1, 2))
X_train_tfidf, X_test_tfidf, vectorizer = fit_and_transform_tfidf(vectorizer, X_train, X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape :", X_test_tfidf.shape)

## 6. Train and Benchmark Five Machine Learning Models
We compare:
1. Logistic Regression
2. Multinomial Naive Bayes
3. Support Vector Machine (LinearSVC Calibrated)
4. Random Forest Classifier
5. Gradient Boosting Classifier

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB(alpha=0.1),
    "Support Vector Machine (SVM)": CalibratedClassifierCV(LinearSVC(random_state=42, C=1.0, dual=False), cv=3),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=35, max_features="sqrt", random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=50, max_depth=3, max_features=60, subsample=0.8, random_state=42)
}

results = []
for name, clf in models.items():
    print(f"Training {name}...")
    clf.fit(X_train_tfidf, y_train)
    res = evaluate_single_model(clf, X_test_tfidf, y_test, model_name=name)
    results.append(res)

comp_df = generate_comparison_report(results, reports_dir="../reports")
print("\n--- MODEL COMPARISON TABLE ---")
print(comp_df.to_string(index=False))

## 7. Comparative Metric Visualizations

In [ ]:
# Plot Accuracy and F1 Score comparisons
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=comp_df, x="Model", y="Accuracy", ax=axes[0], palette="Blues_r")
axes[0].set_title("Model Accuracy Comparison", fontsize=13, weight="bold")
axes[0].set_ylim(0.7, 1.0)
axes[0].tick_params(axis="x", rotation=25)

sns.barplot(data=comp_df, x="Model", y="F1 Score", ax=axes[1], palette="Greens_r")
axes[1].set_title("Model F1 Score Comparison (Primary Criterion)", fontsize=13, weight="bold")
axes[1].set_ylim(0.7, 1.0)
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

## 8. Combined ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))
for res in results:
    y_prob = res.get("y_prob")
    if y_prob is not None and not np.isnan(res["ROC-AUC"]):
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.plot(fpr, tpr, lw=2, label=f"{res['Model']} (AUC = {res['ROC-AUC']:.4f})")

plt.plot([0, 1], [0, 1], "k--", lw=2, label="Random Guess (0.5000)")
plt.xlabel("False Positive Rate", fontsize=11)
plt.ylabel("True Positive Rate", fontsize=11)
plt.title("ROC Curves on Test Dataset", fontsize=13, weight="bold")
plt.legend(loc="lower right")
plt.show()

## 9. Interactive Inference Demonstration

In [ ]:
from src.prediction import NewsPredictor

predictor = NewsPredictor(model_path="../models/best_model.pkl", vectorizer_path="../models/tfidf_vectorizer.pkl")
sample_headline = "Federal Reserve Announces Benchmark Interest Rate Decision"
sample_body = "The central bank held rates steady today following its two-day committee meeting, citing stable economic growth and cooling inflation."

prediction_result = predictor.predict(sample_headline, sample_body)
print("Prediction Output:")
for k, v in prediction_result.items():
    print(f"  {k}: {v}")